<a href="https://colab.research.google.com/github/kilianodonell-cmd/Q3_Durban/blob/main/Field_Map_Deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive only when running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    print('Colab environment detected. Google Drive mounted.')
else:
    print('Local environment detected. Skipping Google Drive mount.')

print('Setup complete.')

Mounted at /content/drive
Setup complete.


In [2]:
import os, json, numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import folium
from folium import LayerControl
from PIL import Image
import base64, io
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# ============================================================
# CELL 1 - Setup
# Field Map - Housing Suitability
# Mzinyati Stream Catchment, eThekwini Municipality
# ============================================================

# Load config (created by MCA notebook)
# Priority order:
# 1) FIELD_MAP_CONFIG_PATH environment variable
# 2) Local repo outputs
# 3) Colab default path
candidate_config_paths = []

env_config = os.environ.get('FIELD_MAP_CONFIG_PATH', '').strip()
if env_config:
    candidate_config_paths.append(env_config)

# Local workspace candidate path
candidate_config_paths.append(os.path.join(os.getcwd(), 'outputs', 'field_map_config.json'))

# Colab default candidate path
candidate_config_paths.append('/content/drive/MyDrive/Durban/outputs/field_map_config.json')

CONFIG_PATH = None
for path in candidate_config_paths:
    if os.path.exists(path):
        CONFIG_PATH = path
        break

if CONFIG_PATH is None:
    raise FileNotFoundError(
        'Could not find field_map_config.json. Set FIELD_MAP_CONFIG_PATH or generate outputs from Durban_MCA.ipynb.'
    )

with open(CONFIG_PATH) as f:
    config = json.load(f)

OUTPUT_ROOT = config['OUTPUT_ROOT']
TARGET_CRS = config['TARGET_CRS']
SCENARIOS = config['SCENARIOS']
SUIT_COLORS = config['SUIT_COLORS']
SUIT_LABELS = config['SUIT_LABELS']
AOI_FIELD = config['AOI_FIELD']
AOI_VALUE = config['AOI_VALUE']
CATCHMENTS_PATH = config['CATCHMENTS_PATH']

RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')
BUILDINGS_PATH = os.path.join(OUTPUT_ROOT, 'outputs', 'at_risk_buildings.gpkg')
CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')

print('=' * 60)
print('FIELD MAP SETUP')
print('=' * 60)
print(f'  Config path: {CONFIG_PATH}')
print(f'  Output root: {OUTPUT_ROOT}')
print(f'  Scenarios:   {list(SCENARIOS.keys())}')

# Check that required files exist
print('\n  Checking files...')
missing = []

for scenario in SCENARIOS:
    raster_path = os.path.join(RASTERS_DIR, f'suitability_{scenario}.tif')
    if os.path.exists(raster_path):
        print(f'    ok suitability_{scenario}.tif')
    else:
        print(f'    missing suitability_{scenario}.tif')
        missing.append(raster_path)

constraint_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
if os.path.exists(constraint_path):
    print('    ok constraint_mask.tif')
else:
    print('    missing constraint_mask.tif')
    missing.append(constraint_path)

if os.path.exists(BUILDINGS_PATH):
    print('    ok at_risk_buildings.gpkg')
else:
    print('    missing at_risk_buildings.gpkg')
    missing.append(BUILDINGS_PATH)

if missing:
    print(f'\n  Warning: {len(missing)} files missing. Run Durban_MCA.ipynb first.')
else:
    print('\n  All files ready. Proceed to the map cell.')

print('=' * 60)

FIELD MAP SETUP
  Output root: /content/drive/MyDrive/Durban/outputs
  Scenarios:   ['hazard_focused', 'balanced', 'infrastructure_focused']

  Checking files...
    ✓ suitability_hazard_focused.tif
    ✓ suitability_balanced.tif
    ✓ suitability_infrastructure_focused.tif
    ✓ constraint_mask.tif
    ✓ at_risk_buildings.gpkg

  ✓ All files ready. Proceed to CELL 2.


In [ ]:
# ============================================================
# FIELD MAP DEEP - INTERACTIVE POC MAP
# - Dynamic scenario layers from MCA config (no hardcoding)
# - Constraint filter layers (in-constraint / non-constraint)
# - Draw tool for candidate polygon selection
# ============================================================

import os
import json
import numpy as np
import folium
from folium import plugins
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
from IPython.display import IFrame, display

print('=' * 60)
print('FIELD MAP DEEP POC')
print('=' * 60)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
AT_RISK_PATH = os.path.join(OUTPUT_ROOT, 'outputs', 'at_risk_buildings.gpkg')
CONSTRAINT_RASTER = os.path.join(RASTERS_DIR, 'constraint_mask.tif')

if not os.path.exists(AT_RISK_PATH):
    raise FileNotFoundError(f'Missing buildings layer: {AT_RISK_PATH}')
if not os.path.exists(CONSTRAINT_RASTER):
    raise FileNotFoundError(f'Missing constraint raster: {CONSTRAINT_RASTER}')

buildings = gpd.read_file(AT_RISK_PATH)
if 'in_constraint' not in buildings.columns:
    raise ValueError("Expected 'in_constraint' column in at_risk_buildings.gpkg")

# Validate that scenario columns exist in the buildings layer.
scenario_keys = list(SCENARIOS.keys())
missing_cols = []
for scenario in scenario_keys:
    dcol = f'display_score_{scenario}'
    rcol = f'raw_score_{scenario}'
    if dcol not in buildings.columns:
        missing_cols.append(dcol)
    if rcol not in buildings.columns:
        missing_cols.append(rcol)

if missing_cols:
    missing_msg = '\n'.join([f'  - {c}' for c in missing_cols])
    raise ValueError(
        'Scenario score columns missing from building outputs. '\
        'Re-run Durban_MCA.ipynb and check schema:\n' + missing_msg
    )

buildings = buildings.to_crs(4326)
buildings['geometry'] = buildings.geometry.simplify(0.00001, preserve_topology=True)

constraint_buildings = buildings[buildings['in_constraint'] == True].copy()
non_constraint_buildings = buildings[buildings['in_constraint'] != True].copy()

# Load AOI boundary
catchments = gpd.read_file(CATCHMENTS_PATH)
aoi = catchments[catchments[AOI_FIELD] == AOI_VALUE].copy().to_crs(4326)
aoi['geometry'] = aoi.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# BUILD CONSTRAINT AREA FROM RASTER
# ------------------------------------------------------------
constraint_mask_gdf = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

with rasterio.open(CONSTRAINT_RASTER) as src:
    mask_data = src.read(1)
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

    print('Constraint raster unique values:', np.unique(mask_data))

    valid_mask = (mask_data != nodata) if nodata is not None else np.ones(mask_data.shape, dtype=bool)

    polygons = []
    for geom, value in shapes(mask_data, mask=valid_mask, transform=transform):
        if value == 0:
            polygons.append(shape(geom))

    if polygons:
        constraint_mask_gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs).to_crs(4326)
        constraint_mask_gdf['geometry'] = constraint_mask_gdf.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# MAP BASE
# ------------------------------------------------------------
center = aoi.union_all().centroid
bounds = aoi.total_bounds

print(f'AOI: {AOI_VALUE}')
print(f'Total buildings: {len(buildings):,}')
print(f'In-constraint buildings: {len(constraint_buildings):,}')
print(f'Non-constraint buildings: {len(non_constraint_buildings):,}')
print(f'Constraint polygons: {len(constraint_mask_gdf):,}')

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=14,
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OSM</a> &copy; CARTO',
    control_scale=True,
)

# AOI boundary
folium.GeoJson(
    aoi,
    name='AOI boundary',
    style_function=lambda feat: {
        'fillColor': 'none',
        'color': '#111111',
        'weight': 2,
        'dashArray': '5, 5',
    },
).add_to(m)

# Constraint area layer
if len(constraint_mask_gdf) > 0:
    folium.GeoJson(
        constraint_mask_gdf,
        name='Filter: Constraint area',
        style_function=lambda feat: {
            'fillColor': '#dc2626',
            'color': '#991b1b',
            'weight': 0.7,
            'fillOpacity': 0.28,
        },
        tooltip='Constraint area',
        show=True,
    ).add_to(m)

# In-constraint buildings layer
if len(constraint_buildings) > 0:
    constraint_popup_fields = ['in_constraint']
    constraint_popup_aliases = ['In constraint:']

    # Include one scenario score in popup for quick context.
    first_scenario = scenario_keys[0]
    constraint_popup_fields.extend([f'display_score_{first_scenario}', f'raw_score_{first_scenario}'])
    constraint_popup_aliases.extend([
        f'Display score ({first_scenario}):',
        f'Raw score ({first_scenario}):',
    ])

    folium.GeoJson(
        constraint_buildings,
        name='Filter: Buildings in constraint',
        style_function=lambda feat: {
            'fillColor': '#7f1a1a',
            'color': '#3f0a0a',
            'weight': 0.25,
            'fillOpacity': 0.9,
        },
        popup=folium.GeoJsonPopup(
            fields=constraint_popup_fields,
            aliases=constraint_popup_aliases,
            localize=True,
            labels=True,
        ),
        show=True,
    ).add_to(m)

# ------------------------------------------------------------
# DYNAMIC SCENARIO LAYERS (NON-CONSTRAINT BUILDINGS)
# ------------------------------------------------------------
for idx, scenario in enumerate(scenario_keys):
    display_col = f'display_score_{scenario}'
    raw_col = f'raw_score_{scenario}'

    def style_for_scenario(feature, display_field=display_col):
        score = feature['properties'].get(display_field, None)
        if score in [1, 2, 3, 4, 5]:
            return {
                'fillColor': SUIT_COLORS[int(score) - 1],
                'color': '#2d2d2d',
                'weight': 0.2,
                'fillOpacity': 0.85,
            }
        return {
            'fillColor': '#9e9e9e',
            'color': '#2d2d2d',
            'weight': 0.2,
            'fillOpacity': 0.85,
        }

    layer_name = f'Scenario: {scenario} (buildings not in constraint)'

    folium.GeoJson(
        non_constraint_buildings,
        name=layer_name,
        style_function=style_for_scenario,
        popup=folium.GeoJsonPopup(
            fields=['in_constraint', display_col, raw_col],
            aliases=['In constraint:', 'Display score:', 'Raw score:'],
            localize=True,
            labels=True,
        ),
        tooltip=folium.GeoJsonTooltip(
            fields=[display_col],
            aliases=['Class:'],
            sticky=False,
        ),
        show=True if idx == 0 else False,
    ).add_to(m)

# ------------------------------------------------------------
# MAP TOOLS
# ------------------------------------------------------------
plugins.Fullscreen().add_to(m)
plugins.MeasureControl(position='topleft', primary_length_unit='meters').add_to(m)
plugins.MousePosition(position='topright').add_to(m)
plugins.Draw(
    export=True,
    filename='candidate_area.geojson',
    position='topleft',
    draw_options={
        'polyline': False,
        'rectangle': True,
        'circle': False,
        'marker': False,
        'circlemarker': False,
    },
    edit_options={'edit': True, 'remove': True},
).add_to(m)

# ------------------------------------------------------------
# LEGEND + GUIDANCE + QUICK SUMMARY
# ------------------------------------------------------------
summary_rows = []
for scenario in scenario_keys:
    col = f'display_score_{scenario}'
    counts = non_constraint_buildings[col].value_counts(dropna=False).to_dict()
    row = f"<tr><td>{scenario}</td><td>{int(counts.get(1, 0))}</td><td>{int(counts.get(2, 0))}</td><td>{int(counts.get(3, 0))}</td><td>{int(counts.get(4, 0))}</td><td>{int(counts.get(5, 0))}</td></tr>"
    summary_rows.append(row)

legend_html = f"""
<div style="position: fixed; bottom: 18px; left: 18px; z-index: 9999; background: white; border: 1px solid #d1d5db; border-radius: 6px; padding: 10px 12px; font-size: 12px; max-width: 350px;">
  <div style="font-weight: 700; margin-bottom: 6px;">Housing Suitability POC</div>
  <div style="margin-bottom: 6px;">Use <b>Layer Control</b> to select one scenario layer and toggle constraint filters.</div>
  <div style="margin-bottom: 6px;">Draw a polygon/rectangle to mark a candidate area and export it as GeoJSON.</div>
  <div><b>Suitability Classes (Non-Constraint)</b></div>
  <div><span style="background:{SUIT_COLORS[0]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[0]}</div>
  <div><span style="background:{SUIT_COLORS[1]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[1]}</div>
  <div><span style="background:{SUIT_COLORS[2]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[2]}</div>
  <div><span style="background:{SUIT_COLORS[3]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[3]}</div>
  <div><span style="background:{SUIT_COLORS[4]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[4]}</div>
  <hr style="margin: 6px 0;">
  <div><span style="background:#dc2626;">&nbsp;&nbsp;&nbsp;</span> Constraint area</div>
  <div><span style="background:#7f1a1a;">&nbsp;&nbsp;&nbsp;</span> Buildings in constraint</div>
  <hr style="margin: 6px 0;">
  <div><b>Current Input Data Summary (non-constraint count by class)</b></div>
  <table style="border-collapse: collapse; font-size: 11px;">
    <tr><th style="text-align:left; padding-right:8px;">Scenario</th><th>1</th><th>2</th><th>3</th><th>4</th><th>5</th></tr>
    {''.join(summary_rows)}
  </table>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

# ------------------------------------------------------------
# SAVE OUTPUTS (LOCAL POC + SHARING CANDIDATE)
# ------------------------------------------------------------
output_dir = os.path.join(OUTPUT_ROOT, 'outputs')
os.makedirs(output_dir, exist_ok=True)
output_map = os.path.join(output_dir, 'field_map_deep_poc.html')
m.save(output_map)

print(f'\nMap saved: {output_map}')
print('Tip: publish this HTML for online proof-of-concept sharing.')
display(IFrame(output_map, width='100%', height='680'))

FINAL MAP
Unique values in constraint raster: [0 1]
AOI: Mzinyati Stream
Total buildings: 22,196
Constraint buildings: 4,268
Non-constraint buildings: 17,928
Constraint polygons: 129

✅ Map saved: /content/drive/MyDrive/Durban/outputs/outputs/final_map.html
